
**Topic:** *Evaluating AI-Based Streamflow Forecasts for Improving Electricity Supply Reliability During Droughts in Ecuador*.


## 1. Libraries and configuration

In [ ]:

# 1. LIBRARIES AND CONFIGURATION


import re
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBRegressor


FILE_PATH = Path(
    r"C:\Users\DETPC\Downloads\proyecto\Supplementary File S1.xlsx"
)

SHEET_NAME = 0
DATE_COL = None
YEAR_COL = None
MONTH_COL = None
FLOW_COL = None

# Predict one month ahead
FORECAST_HORIZON = 1

OUTPUT_DIR = FILE_PATH.parent / "results_paute"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("File path:", FILE_PATH)
print("File exists:", FILE_PATH.exists())

if not FILE_PATH.is_file():
    raise FileNotFoundError(
        f"Excel file not found: {FILE_PATH}"
    )
excel = pd.ExcelFile(FILE_PATH)
print("\nAvailable sheets:")
print(excel.sheet_names)

df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
df = pd.read_excel()
FILE_PATH,
sheet_name=SHEET_NAME
print("\nDataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
print(df.head().to_string(index=False))


File path: C:\Users\DETPC\Downloads\proyecto\Supplementary File S1.xlsx
File exists: True

Available sheets:
['Dataset_monthly', 'Quality_control', 'Monthly_climatology', 'Annual_summary', 'Model_selection', 'Parameter_diagnostics', 'Residual_diagnostics', 'Backtest_rolling', 'PI_coverage', 'Prob_verification', 'Threshold_sensitivity', 'Forecast_2026', 'Forecast_2026_2030', 'Forecast_highest_risk', 'Dry_wet_years']


TypeError: read_excel() missing 1 required positional argument: 'io'

In [21]:
print(excel.sheet_names)
print(df.columns.tolist())
print(df.head())

['Dataset_monthly', 'Quality_control', 'Monthly_climatology', 'Annual_summary', 'Model_selection', 'Parameter_diagnostics', 'Residual_diagnostics', 'Backtest_rolling', 'PI_coverage', 'Prob_verification', 'Threshold_sensitivity', 'Forecast_2026', 'Forecast_2026_2030', 'Forecast_highest_risk', 'Dry_wet_years']


NameError: name 'df' is not defined

## 2. Localizar e inspeccionar el Excel

In [ ]:
if FILE_PATH is None:
    roots = [Path('/kaggle/input'), Path('/mnt/data'), Path('.')]
    excel_files = []
    for root in roots:
        if root.exists():
            excel_files.extend(p for p in root.rglob('*.xlsx') if not p.name.startswith('~$'))
    excel_files = sorted(set(excel_files))
    matching = [p for p in excel_files if 'supplementary' in p.name.lower() and 's1' in p.name.lower()]
    if len(matching) == 1:
        path = matching[0]
    else:
        print('Excel files detected:')
        for i, p in enumerate(excel_files):
            print(f'{i}: {p}')
        raise FileNotFoundError(
            'Upload Supplementary File S1.xlsx to Kaggle, then set FILE_PATH to its exact path.'
        )
else:
    path = Path(FILE_PATH)

if not path.exists():
    raise FileNotFoundError(f'Dataset not found: {path}')

book = pd.ExcelFile(path)
print('File:', path)
print('Sheets:', book.sheet_names)
for sheet in book.sheet_names:
    preview = pd.read_excel(path, sheet_name=sheet, nrows=5)
    print(f'\nSheet: {sheet} | columns: {preview.columns.tolist()}')
    print(preview.head(3).to_string(index=False))

raw_sheet = pd.read_excel(path, sheet_name=SHEET_NAME)
print('\nSelected sheet:', SHEET_NAME)
print('Shape:', raw_sheet.shape)
print('Columns:', raw_sheet.columns.tolist())



## 3. Identificar y limpiar caudales observados

In [ ]:

def norm(text):
    text = unicodedata.normalize('NFKD', str(text)).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-z0-9]+', '_', text.lower()).strip('_')


def identify_col(columns, fragments, configured=None, field='column'):
    if configured is not None:
        if configured not in columns:
            raise ValueError(f'{field}: {configured!r} not found. Available: {list(columns)}')
        return configured
    matched = [c for c in columns if any(fragment in norm(c) for fragment in fragments)]
    if len(matched) == 1:
        return matched[0]
    return None

cols = list(raw_sheet.columns)
flow_col = identify_col(cols,
    ['caudal_observado', 'observed_flow', 'streamflow', 'discharge', 'caudal', 'aforo', 'inflow', 'q_m3'],
    configured=FLOW_COL, field='FLOW_COL'
)
# Also accept a column literally called 'Q', but do not guess among several numerical columns.
if flow_col is None and 'q' in [norm(c) for c in cols]:
    flow_col = cols[[norm(c) for c in cols].index('q')]

if flow_col is None:
    raise ValueError(
        'Cannot unambiguously find observed streamflow. Inspect the sheet and set FLOW_COL '
        f'to the correct name (in m3/s). Columns: {cols}'
    )

date_col = identify_col(cols, ['fecha', 'date', 'periodo', 'period'], DATE_COL, 'DATE_COL')
if date_col is not None:
    dates = pd.to_datetime(raw_sheet[date_col], errors='coerce')
else:
    year_col = identify_col(cols, ['year', 'anio', 'ano'], YEAR_COL, 'YEAR_COL')
    month_col = identify_col(cols, ['month', 'mes'], MONTH_COL, 'MONTH_COL')
    if year_col is None or month_col is None:
        raise ValueError(
            'Date not detected. Define DATE_COL, or YEAR_COL and MONTH_COL, '
            f'after inspecting the Excel. Columns: {cols}'
        )
    month_text = raw_sheet[month_col].astype('string').str.strip().str.lower()
    spanish_months = {
        'enero': 1, 'febrero': 2, 'marzo': 3, 'abril': 4, 'mayo': 5, 'junio': 6,
        'julio': 7, 'agosto': 8, 'septiembre': 9, 'setiembre': 9, 'octubre': 10,
        'noviembre': 11, 'diciembre': 12
    }
    month_values = pd.to_numeric(month_text, errors='coerce').fillna(month_text.map(spanish_months))
    dates = pd.to_datetime({
        'year': pd.to_numeric(raw_sheet[year_col], errors='coerce'),
        'month': month_values,
        'day': 1
    }, errors='coerce')

flow = pd.to_numeric(raw_sheet[flow_col], errors='coerce')
monthly = pd.DataFrame({'date': dates, 'flow': flow}).dropna(subset=['date'])
monthly['date'] = monthly['date'].dt.to_period('M').dt.to_timestamp()
monthly = monthly.sort_values('date').set_index('date')

if monthly.index.duplicated().any():
    duplicates = monthly.index[monthly.index.duplicated()].unique().strftime('%Y-%m').tolist()
    raise ValueError(
        f'Duplicate months detected: {duplicates[:8]}. Select the right worksheet or '
        'aggregate only after checking the meaning of duplicate rows.'
    )

monthly = monthly.asfreq('MS')
if monthly['flow'].isna().any():
    missing = monthly.index[monthly['flow'].isna()].strftime('%Y-%m').tolist()
    raise ValueError(f'Missing/invalid monthly flows: {missing[:15]}; audit them before modeling.')
if (monthly['flow'] < 0).any():
    raise ValueError('Negative streamflow found. Review units and source columns.')

print('\nAudit')
print('Observed flow column:', flow_col)
print('Time span:', monthly.index.min(), 'to', monthly.index.max())
print('Total months:', len(monthly))
print('Missing flows:', monthly['flow'].isna().sum())
print(monthly['flow'].describe())



## 4. Features y división cronológica

In [ ]:
features_df = monthly.copy()
for lag in [1, 2, 3, 6, 12]:
    features_df[f'lag_{lag}'] = features_df['flow'].shift(lag)
for window in [3, 6]:
    features_df[f'rolling_mean_{window}'] = (
        features_df['flow'].shift(1).rolling(window=window).mean()
    )
features_df['month_sin'] = np.sin(2*np.pi * features_df.index.month / 12)
features_df['month_cos'] = np.cos(2*np.pi * features_df.index.month / 12)

feature_cols = [
    'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12',
    'rolling_mean_3', 'rolling_mean_6', 'month_sin', 'month_cos'
]
data = features_df.dropna(subset=feature_cols + ['flow'])

train = data.loc[:'2016-12-01'].copy()
val = data.loc['2017-01-01':'2022-12-01'].copy()
test = data.loc['2023-01-01':'2024-12-01'].copy()
trainval = data.loc[:'2022-12-01'].copy()

assert len(train) >= 60, 'Not enough training observations.'
assert len(val) == 72, 'Expected 72 validation months (2017-2022).'
assert len(test) == 24, 'Expected 24 testing months (2023-2024).'

print('\nChronological partition:')
print('Train:', len(train), train.index.min().date(), train.index.max().date())
print('Validation:', len(val), val.index.min().date(), val.index.max().date())
print('Test:', len(test), test.index.min().date(), test.index.max().date())



## 5. Baselines y modelos de IA

In [ ]:

def models_to_compare():
    return {
        'Random Forest': RandomForestRegressor(
            n_estimators=200, max_depth=8, min_samples_leaf=3,
            random_state=42, n_jobs=-1
        ),
        'XGBoost': XGBRegressor(
            n_estimators=250, max_depth=3, learning_rate=0.04,
            subsample=0.9, colsample_bytree=0.9,
            objective='reg:squarederror', random_state=42, n_jobs=2
        ),
    }


def sarima_walk_forward(history_flow, observed_future):
    """One-month-ahead log-SARIMA forecast; update only after the month is observed."""
    history = np.log1p(history_flow.asfreq('MS'))
    model = SARIMAX(
        history, order=(1, 0, 0), seasonal_order=(1, 1, 0, 12),
        trend='n', enforce_stationarity=False, enforce_invertibility=False
    )
    fitted = model.fit(disp=False, maxiter=120)
    predictions = []
    for date, actual in observed_future.items():
        log_forecast = float(fitted.get_forecast(steps=1).predicted_mean.iloc[0])
        predictions.append(max(0., float(np.expm1(log_forecast))))
        new_observation = pd.Series(
            [np.log1p(actual)], index=pd.DatetimeIndex([date]), name='flow'
        )
        fitted = fitted.append(new_observation, refit=False)
    return np.asarray(predictions)


def metrics(actual, pred):
    return {
        'MAE (m3/s)': mean_absolute_error(actual, pred),
        'RMSE (m3/s)': np.sqrt(mean_squared_error(actual, pred))
    }



## 6. Validación: 2017–2022

In [ ]:
validation_predictions = {
    'Seasonal Naive': val['lag_12'].to_numpy(),
    'Persistence': val['lag_1'].to_numpy(),
}
validation_predictions['SARIMA'] = sarima_walk_forward(
    monthly.loc[:'2016-12-01', 'flow'],
    monthly.loc['2017-01-01':'2022-12-01', 'flow']
)

for name, model in models_to_compare().items():
    model.fit(train[feature_cols], train['flow'])
    validation_predictions[name] = np.maximum(0, model.predict(val[feature_cols]))

validation_results = pd.DataFrame([
    {'Model': name, **metrics(val['flow'], p)}
    for name, p in validation_predictions.items()
]).sort_values('MAE (m3/s)').reset_index(drop=True)
print('\nVALIDATION RESULTS (2017-2022):')
print(validation_results.round(3).to_string(index=False))

ai_validation = validation_results[
    validation_results['Model'].isin(['Random Forest', 'XGBoost'])
]
best_ai_name = ai_validation.iloc[0]['Model']
print('\nAI model selected using ONLY validation MAE:', best_ai_name)



## 7. Test: 2023–2024

In [ ]:
predictions = pd.DataFrame(index=test.index)
predictions.index.name = 'date'
predictions['Observed'] = test['flow']
predictions['Seasonal Naive'] = test['lag_12']
predictions['Persistence'] = test['lag_1']
predictions['SARIMA'] = sarima_walk_forward(
    monthly.loc[:'2022-12-01', 'flow'],
    monthly.loc['2023-01-01':'2024-12-01', 'flow']
)

fitted_models = {}
for name, model in models_to_compare().items():
    model.fit(trainval[feature_cols], trainval['flow'])
    predictions[name] = np.maximum(0, model.predict(test[feature_cols]))
    fitted_models[name] = model

test_results = pd.DataFrame([
    {'Model': name, **metrics(predictions['Observed'], predictions[name])}
    for name in ['Seasonal Naive', 'Persistence', 'SARIMA', 'Random Forest', 'XGBoost']
]).sort_values('MAE (m3/s)').reset_index(drop=True)

print('\nTEST RESULTS (2023-2024):')
print(test_results.round(3).to_string(index=False))



## 8. Análisis de meses con caudal bajo

In [ ]:
low_threshold = float(trainval['flow'].quantile(0.20))
low_mask = predictions['Observed'] <= low_threshold
print(f'\nLow-flow threshold (20th percentile of pre-test history): {low_threshold:.3f} m3/s')
print('Low-flow months in test:', int(low_mask.sum()))

lowflow_results = []
for model in ['Seasonal Naive', 'Persistence', 'SARIMA', 'Random Forest', 'XGBoost']:
    lowflow_results.append({
        'Model': model,
        'Low-flow MAE (m3/s)': (
            mean_absolute_error(predictions.loc[low_mask, 'Observed'],
                                predictions.loc[low_mask, model])
            if low_mask.any() else np.nan
        )
    })
lowflow_results = pd.DataFrame(lowflow_results)
print(lowflow_results.round(3).to_string(index=False))



## 9. Exportar tablas y gráficas reales

In [ ]:
validation_results.to_csv(OUTPUT_DIR / 'paute_validation_metrics.csv', index=False)
test_results.to_csv(OUTPUT_DIR / 'paute_test_metrics.csv', index=False)
lowflow_results.to_csv(OUTPUT_DIR / 'paute_lowflow_metrics.csv', index=False)
predictions.to_csv(OUTPUT_DIR / 'paute_test_predictions.csv', index=True)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(predictions.index, predictions['Observed'], marker='o', label='Observed')
ax.plot(predictions.index, predictions['Seasonal Naive'], label='Seasonal Naive')
ax.plot(predictions.index, predictions[best_ai_name], label=f'AI: {best_ai_name}')
ax.set(title='Paute River monthly streamflow: 2023-2024 test',
       xlabel='Date', ylabel='Streamflow (m3/s)')
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'paute_test_forecasts.png', dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(test_results['Model'], test_results['MAE (m3/s)'])
ax.set(title='One-month-ahead forecasting MAE on test set',
       ylabel='MAE (m3/s)')
ax.tick_params(axis='x', rotation=25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'paute_test_mae.png', dpi=160)
plt.show()

with open(OUTPUT_DIR / 'paute_preliminary_results.md', 'w', encoding='utf-8') as f:
    f.write('# 6. Preliminary Results — Paute streamflow forecasting\n\n')
    f.write('Dataset: monthly observed streamflow in Paute, sourced from Zenodo 20213449.\n\n')
    f.write('Target: one-month-ahead streamflow (m3/s).\n\n')
    f.write('Train: through 2016; validation: 2017-2022; test: 2023-2024.\n\n')
    f.write('## Validation results\n\n')
    f.write('```text\n' + validation_results.round(3).to_string(index=False) + '\n```')
    f.write('\n\n## Test results\n\n')
    f.write('```text\n' + test_results.round(3).to_string(index=False) + '\n```')
    f.write(f'\n\nAI model selected on validation: {best_ai_name}.\n')
    f.write('\nThese are forecasting results, not simulated electricity reliability results.\n')

print('\nSAVED FILES IN:', OUTPUT_DIR)
for filename in [
    'paute_validation_metrics.csv', 'paute_test_metrics.csv',
    'paute_lowflow_metrics.csv', 'paute_test_predictions.csv',
    'paute_test_forecasts.png', 'paute_test_mae.png',
    'paute_preliminary_results.md'
]:
    print('-', filename)
